# Coefficient Giving Downloads Pipeline

Coefficient Giving (previously Open Philanthropy) have a JS run website which makes it difficult and inefficient to scrap. However there is a (hidden) download CSV file, which according to [LongerTerm Wiki](https://www.longtermwiki.com/data-sources/coefficient-giving) was updated 2 April 2026, although the most recent grant record appears to be from October 2025. Datasets are available at: https://coefficientgiving.org/wp-content/uploads/Coefficient-Giving-Grants-Archive.csv.

This notebook will:

1. -Download the datafile- Apparently we are not allowed to download the file programatically.
2. Dump the resulting files into the target DATA_INPUT_DIRECTORY
3. Read each file in the directory in
4. Write out a json-lines format file in the OIC datamodel to the designated data output directory. Output location MUST be a directory to allow for batching of the outputs into multiple files

In [7]:
from pathlib import Path
import csv
import json
from attrs import asdict
import polars as pl
from datetime import datetime, date
import requests

from oic_scrape.items import AwardParticipant, AwardItem


In [12]:
DATA_INPUT_DIRECTORY = Path("../data/wellcome/input_data")
FUNDER_ORG_NAME = "Coefficient Giving"
FUNDER_ROR_ID = "https://ror.org/030prv062"
OUTPUT_LOCATION = Path("../data/wellcome")
OUTPUT_FORMAT = "jsonl"
DOWNLOADED_FILE_NAME = "Coefficient-Giving-Grants-Archive.csv"

In [10]:
# Download data
# cg_url = "https://coefficientgiving.org/wp-content/uploads/Coefficient-Giving-Grants-Archive.csv"
#
# with requests.get(cg_url,
#                   headers = {
#                       "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
#                       "referer": "https://www.longtermwiki.com/data-sources/coefficient-giving",
#                   },
#                   stream=True) as response:
#         response.raise_for_status()
#
#         with open( DATA_INPUT_DIRECTORY / DOWNLOADED_FILE_NAME, "wb") as file:
#             for chunk in response.iter_content(chunk_size=8192):
#                 file.write(chunk)
#             file.flush()


In [17]:
# Process Coefficient Giving Summary View
awards = []
csv_file = DATA_INPUT_DIRECTORY / DOWNLOADED_FILE_NAME
with open(csv_file, "r") as f:
    reader = csv.DictReader(f)
    for grant in reader:
        grant_id = f'coefficientgiving::{grant.get("Organization Name").replace(" ", "_").lower()}_{grant.get("Date").replace(" ", "").lower()}'
        start_date = grant.get('Date', '')
        amount = grant.get('Amount', '')
        award = AwardItem(
            _crawled_at=datetime.now(),
            source='https://coefficientgiving.org/wp-content/uploads/Coefficient-Giving-Grants-Archive.csv',
            grant_id=grant_id,
            funder_org_name=FUNDER_ORG_NAME,
            funder_org_ror_id=FUNDER_ROR_ID,
            program_of_funder=grant.get('Focus Area', ''),
            grant_title=grant.get('Grant', ''),
            grant_description=grant.get('Details', ''),
            recipient_org_name=grant.get('Organization Name', '') ,
            grant_year=int(start_date.split(' ')[1]),
            grant_start_date=datetime.strptime(f'1 {start_date}', "%d %B %Y").date(),
            award_amount=float(amount.removeprefix('$').replace(',','')) if amount != '' else None,
            award_currency='USD',
            award_amount_usd=float(amount.removeprefix('$').replace(',','')) if amount != '' else None,
            _award_schema_version="0.1.0",
            raw_source_data = json.dumps(grant),
        )
        awards.append(asdict(award))


In [18]:
pl.DataFrame(awards).write_ndjson(OUTPUT_LOCATION / f'coefficientgiving.jsonl')